# Convert Celsius values to Fahrenheit using CUDA.

In [1]:
import numpy as np
from numba import cuda
import time

# CUDA Kernel
@cuda.jit
def celsius_to_fahrenheit(celsius, fahrenheit):

    idx = cuda.grid(1)

    if idx < celsius.size:
        fahrenheit[idx] = (
            celsius[idx] * 9.0 / 5.0
        ) + 32.0


# ==========================
# Generate Dataset
# ==========================

N = 10_000_000

# Random Celsius values
# Range: -100°C to 100°C

celsius = np.random.uniform(
    -100.0,
    100.0,
    N
).astype(np.float32)

fahrenheit = np.empty_like(celsius)

print(f"Dataset Size : {N:,} values")
print(f"Memory Usage : {celsius.nbytes/1024/1024:.2f} MB")

# ==========================
# Copy to GPU
# ==========================

d_celsius = cuda.to_device(celsius)
d_fahrenheit = cuda.device_array_like(fahrenheit)

# ==========================
# CUDA Configuration
# ==========================

threads_per_block = 256

blocks_per_grid = (
    N + threads_per_block - 1
) // threads_per_block

# ==========================
# Execute Kernel
# ==========================

start = time.time()

celsius_to_fahrenheit[
    blocks_per_grid,
    threads_per_block
](
    d_celsius,
    d_fahrenheit
)

cuda.synchronize()

gpu_time = time.time() - start

# ==========================
# Copy Result Back
# ==========================

fahrenheit = d_fahrenheit.copy_to_host()

# ==========================
# Display Results
# ==========================

print("\nSample Results:")
for i in range(10):
    print(
        f"{celsius[i]:7.2f} °C  ->  "
        f"{fahrenheit[i]:7.2f} °F"
    )

print(f"\nGPU Execution Time: {gpu_time:.6f} seconds")

Dataset Size : 10,000,000 values
Memory Usage : 38.15 MB

Sample Results:
  82.37 °C  ->   180.27 °F
 -80.20 °C  ->  -112.36 °F
 -64.06 °C  ->   -83.30 °F
 -81.40 °C  ->  -114.52 °F
 -29.00 °C  ->   -20.20 °F
  12.85 °C  ->    55.13 °F
 -37.74 °C  ->   -35.93 °F
  80.81 °C  ->   177.45 °F
  89.90 °C  ->   193.83 °F
  36.38 °C  ->    97.48 °F

GPU Execution Time: 2.465202 seconds
